# clinKriya GRPO Training with Unsloth + TRL

Adapted from the `OpenEnv_gpt_oss_(20B)_Reinforcement_Learning_2048_Game_BF16` notebook.

**What's swapped:** reward function + environment step (clinKriya FHIR medical tasks).  
**What's verbatim:** model loading, LoRA setup, GRPOConfig.

Model: **gpt-oss-20B BF16** | Env: **MedAgentBench FHIR** | RL: **GRPO**

# Installation

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy; get_numpy = f"numpy=={numpy.__version__}"
    except: get_numpy = "numpy"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {get_numpy} torchvision bitsandbytes "transformers==4.56.2" trackio \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth trackio
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

In [ ]:
%%capture
import sys
from pathlib import Path

REPO_DIR = Path('./clinKriya')
if not REPO_DIR.exists():
    !git clone https://github.com/ananya173147/clinKriya.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

sys.path.insert(0, str(REPO_DIR))


# Load gpt-oss-20B (verbatim)

> `max_seq_length = 768` — medical prompts with FHIR context can approach this limit.  
> Increase to `1024` or `2048` here and in GRPOConfig if you see truncation warnings.


In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 768 # Can increase for longer RL output
lora_rank = 4        # Larger rank = smarter, but slower
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gpt-oss-20b-BF16",
    load_in_4bit = False,
    max_seq_length = max_seq_length,
)

# LoRA setup (verbatim)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank*2, # *2 speeds up training
    use_gradient_checkpointing = "unsloth", # Reduces memory usage
    random_state = 3407,
)

# clinKriya Environment

Load `MedAgentTrainEnv`, `reward_func`, and `build_dataset` from `train.py`.
- **`MedAgentTrainEnv`** — one FHIR episode per rollout (replaces the 2048 OpenEnv step)
- **`reward_func`** — shaped reward from `compute_shaped_reward` (replaces `function_works` / `no_cheating` / `strategy_succeeds`)


In [ ]:
import importlib.util
from pathlib import Path

REPO = Path('./clinKriya')

# Load train.py directly — avoids installing openenv-core
spec = importlib.util.spec_from_file_location(
    'train', REPO / 'medagentbench_env' / 'train.py'
)
train_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(train_mod)

MedAgentTrainEnv = train_mod.MedAgentTrainEnv
reward_func      = train_mod.reward_func
build_dataset    = train_mod.build_dataset

DATA_DIR = REPO / 'medagentbench_env' / 'data'

# Override module-level paths to cloned data
train_mod._DATA_DIR           = DATA_DIR
train_mod._CACHE_PATH         = DATA_DIR / 'fhir_cache.json'
train_mod._SYSTEM_PROMPT_PATH = DATA_DIR / 'new_system.txt'

# Pre-load shared FHIR cache (done once, reused across episodes)
train_mod._get_mock_fhir()
tasks = train_mod._get_tasks()
print(f'FHIR cache loaded | {len(tasks)} tasks available')


# Build Training Dataset

Each row is a `{prompt: [{role, content}, ...]}` dict.  
Set `NUM_TASKS` to a small number (e.g. `5`) for a quick smoke-test.


In [ ]:
NUM_TASKS = None  # Set to e.g. 5 for a smoke-test; None = all 90 tasks

dataset = build_dataset(DATA_DIR, num_tasks=NUM_TASKS)

# gpt-oss uses reasoning_effort to control thinking budget.
# 'low' keeps completions short and training fast.
dataset = dataset.map(lambda x: {'reasoning_effort': 'low'})

print(f'Dataset: {len(dataset)} tasks')
print(f'Columns: {dataset.column_names}')

# Compute maximum prompt length so GRPOConfig can derive max_completion_length
maximum_length = max(
    len(tokenizer.apply_chat_template(
        row['prompt'],
        add_generation_prompt=True,
        reasoning_effort='low',
    ))
    for row in dataset
)
print(f'Maximum prompt length: {maximum_length} tokens (max_seq_length={max_seq_length})')
if maximum_length > max_seq_length - 64:
    print('WARNING: prompts are very close to max_seq_length — '
          'consider increasing max_seq_length in the model load cell.')


# Train the model (GRPOConfig verbatim)

GRPOConfig is **unchanged** from the BF16 notebook.


In [ ]:
max_prompt_length = maximum_length + 1 # + 1 just in case!
max_completion_length = max_seq_length - max_prompt_length

from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    temperature = 1.0,
    learning_rate = 5e-5,
    weight_decay = 0.001,
    warmup_ratio = 0.1,
    lr_scheduler_type = "linear",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 1, # Increase to 4 for smoother training
    num_generations = 2, # Decrease if out of memory
    max_prompt_length = max_prompt_length,
    max_completion_length = max_completion_length,
    # num_train_epochs = 1, # Set to 1 for a full training run
    max_steps = 600,
    save_steps = 100,
    report_to = "none", # Can use Weights & Biases, TrackIO
    output_dir = "outputs",

    # For optional training + evaluation
    # fp16_full_eval = True,
    # per_device_eval_batch_size = 4,
    # eval_accumulation_steps = 1,
    # eval_strategy = "steps",
    # eval_steps = 1,
)

### Trainer

Two changes from the 2048 trainer:
1. `reward_funcs = reward_func` — single clinKriya shaped-reward function
2. `environment_factory = MedAgentTrainEnv` — TRL creates one FHIR env per rollout


In [ ]:
# For optional training + evaluation
# new_dataset = dataset.train_test_split(test_size = 0.1)

trainer = GRPOTrainer(
    model              = model,
    processing_class   = tokenizer,
    # Single reward function replaces [function_works, no_cheating, strategy_succeeds]
    reward_funcs       = reward_func,
    args               = training_args,
    train_dataset      = dataset,
    # environment_factory: TRL spins up one MedAgentTrainEnv per rollout.
    # Replaces the OpenEnv / launch_openenv call in the 2048 notebook.
    environment_factory = MedAgentTrainEnv,

    # For optional training + evaluation
    # train_dataset = new_dataset['train'],
    # eval_dataset  = new_dataset['test'],
)


In [ ]:
trainer.train()

<a name="Save"></a>
### Saving to float16 or `MXFP4`

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `mxfp4` for MXFP4 (OpenAI's GPT-OSS native precision). We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
# Merge and push to hub in mxfp4 4bit format
if False:
    model.save_pretrained_merged("finetuned_model", tokenizer, save_method = "mxfp4")
if False:
    model.push_to_hub_merged("repo_id/repo_name", tokenizer, token = "hf...", save_method = "mxfp4")

# Merge and push to hub in 16bit
if False:
    model.save_pretrained_merged("finetuned_model", tokenizer, save_method = "merged_16bit")
if False: # Pushing to HF Hub
    model.push_to_hub_merged("hf/gpt-oss-finetune", tokenizer, save_method = "merged_16bit", token = "")